# 01: Classifier (GPT-4o-mini)

OpenAI parallel pipeline — classifies headlines from the shared clean dataset using `gpt-4o-mini`. Outputs are written locally under `EF-02-openai/data/processed/`.

## Section 1: Setup

In [1]:
import pandas as pd
import json
import time
import re
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv("../../EF-02/.env")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gpt-4o-mini"
BATCH_SIZE = 25
SLEEP_SEC  = 0.15  # OpenAI paid tier — much faster than Groq free tier

VALID_CATEGORIES = {
    "Forex", "Policy", "Banking", "Trade",
    "Agriculture", "Energy", "Transport", "Investment",
    "Markets", "Tourism", "Inflation"
}
VALID_SENTIMENTS = {"Positive", "Negative", "Neutral"}

## Section 2: Prompt

In [2]:
SYSTEM_PROMPT = """You are a financial news classifier for Tanzania.
You return ONLY a valid JSON array. No preamble, no explanation, no markdown fences.
Every object in the array must be complete and the array must be properly closed with ]."""


def build_prompt(batch, include_reason=False):
    lines = "\n".join(
        f"{i+1}. {item['headline']}"
        for i, item in enumerate(batch)
    )

    reason_field = ""
    if include_reason:
        reason_field = '\nAdd "reason": 4-6 words explaining your decision (debugging only).'

    return f"""You classify Tanzanian financial headlines. Output ONLY a valid JSON array.

Schema: [{{"pos":1,"relevant":true,"category":"Forex","sentiment":"Positive"}}, ...]{reason_field}

STEP 0 — RELEVANCE:
relevant true → Tanzania economy, finance, business, trade, banking, currency, energy, agriculture
relevant false → sports, crime, entertainment, foreign-only news, opinion/lifestyle; if false set category/sentiment null

STEP 1 — pick ONE category:
Forex(shilling,dollar,reserves) | Policy(BOT,IMF,budget,tax,debt,GDP,NBS,rates) | Banking(banks,loans,fintech,insurance) | Trade(imports,exports,tariffs,ports,AfCFTA) | Agriculture(crops,farming,food) | Energy(TANESCO,fuel,power) | Transport(SGR,rail,roads,logistics) | Investment(FDI,factories,PPP,crowdfunding) | Markets(DSE,CMSA,equity,bonds,turnover) | Tourism(hotels,arrivals) | Inflation(CPI,food/cement price spikes)

Disambiguation: DSE/CMSA→Markets | BOT rate/GDP→Policy | food/cement price spike→Inflation | hotels→Tourism

STEP 2 — SENTIMENT (economic outcome not tone):
Positive=growth,profit up,shilling firms,easing inflation,launches | Negative=decline,loss,miss target,weakening,rising inflation | Neutral=steady,guidelines,reviews,no clear direction
Falling inflation=Positive. Rising inflation=Negative.

Headlines:
{lines}"""

## Section 3: Test Run
Runs on a sample with `reason` off by default. Use `include_reason=True` only when inspecting individual decisions.

In [3]:
def classify_batch(batch, batch_num, total_batches, include_reason=False, max_retries=3):
    prompt = build_prompt(batch, include_reason=include_reason)
    raw = ""

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt}
                ],
                temperature=0,
                max_tokens=2500
            )

            raw   = response.choices[0].message.content.strip()
            clean = re.sub(r'```(?:json)?|```', '', raw).strip()
            parsed = json.loads(clean)

            if not isinstance(parsed, list):
                raise ValueError(f"Expected list, got {type(parsed)}")
            if len(parsed) != len(batch):
                raise ValueError(f"Got {len(parsed)} results for {len(batch)} headlines")

            for i, result in enumerate(parsed):
                result['id'] = batch[i]['id']
                result.pop('pos', None)

            print(f"  Batch {batch_num}/{total_batches} — OK ({len(parsed)} results)", end="\r", flush=True)
            return parsed

        except json.JSONDecodeError:
            tail = raw[-80:] if len(raw) > 80 else raw
            print(f"  Batch {batch_num} attempt {attempt}: JSON cut off. Ends: ...{repr(tail)}")
            time.sleep(2)

        except ValueError as e:
            print(f"  Batch {batch_num} attempt {attempt}: Validation error — {e}")
            time.sleep(2)

        except Exception as e:
            print(f"  Batch {batch_num} attempt {attempt}: API error — {e}")
            time.sleep(5)

    print(f"  Batch {batch_num}: FAILED after {max_retries} attempts. Filling with None.")
    return [
        {"id": item['id'], "relevant": None, "category": None, "sentiment": None}
        for item in batch
    ]


def run_classifier(input_csv, output_csv, batch_size=BATCH_SIZE,
                   test_size=None, include_reason=False):
    df = pd.read_csv(input_csv, engine='python', on_bad_lines='warn')
    print(f"Loaded: {len(df)} rows from {input_csv}")

    if test_size and test_size < len(df):
        df = df.sample(n=test_size, random_state=42).reset_index(drop=True)
        print(f"Sampled: {len(df)} rows for testing")
    else:
        df = df.reset_index(drop=True)
        print(f"Running on full dataset: {len(df)} rows")

    df['id'] = df.index

    rows    = df[['id', 'headline']].to_dict('records')
    batches = [rows[i:i+batch_size] for i in range(0, len(rows), batch_size)]
    total   = len(batches)
    print(f"\n{total} batches of up to {batch_size} headlines each")
    print(f"Estimated time: ~{total * SLEEP_SEC / 60:.1f} minutes\n")

    all_results = []
    for batch_num, batch in enumerate(batches, start=1):
        results = classify_batch(batch, batch_num, total, include_reason=include_reason)
        all_results.extend(results)
        if batch_num < total:
            time.sleep(SLEEP_SEC)

    df_results = pd.DataFrame(all_results)
    df_out     = df.merge(df_results, on='id', how='left').drop(columns=['id'])

    # Print summary
    total_rows = len(df_out)
    n_rel      = df_out['relevant'].sum()
    n_failed   = df_out['relevant'].isna().sum()

    print(f"\n{'='*50}")
    print(f"DONE")
    print(f"{'='*50}")
    print(f"  Total rows   : {total_rows}")
    print(f"  Relevant     : {int(n_rel)} ({n_rel/total_rows*100:.1f}%)")
    print(f"  Irrelevant   : {int(total_rows - n_rel)} ({(total_rows - n_rel)/total_rows*100:.1f}%)")
    if n_failed:
        print(f"  Failed rows  : {int(n_failed)} — run Section 5 retry cell to fix")

    print(f"\n  Category breakdown (relevant only):")
    for cat, count in df_out[df_out['relevant'] == True]['category'].value_counts().items():
        print(f"    {cat:<14} {count}")

    print(f"\n  Sentiment breakdown (relevant only):")
    for sent, count in df_out[df_out['relevant'] == True]['sentiment'].value_counts().items():
        print(f"    {sent:<14} {count}")

    df_out.to_csv(output_csv, index=False)
    print(f"\nSaved to {output_csv}")

    return df_out

In [5]:
# Test run — 300 headlines, batch size 10, no reason field
df_test = run_classifier(
    input_csv      = "../../EF-02/data/processed/tz_headlines_clean.csv",
    output_csv     = "../data/processed/tz_headlines_test.csv",
    test_size      = 300,
    include_reason = False
)

Loaded: 6235 rows from ../../EF-02/data/processed/tz_headlines_clean.csv
Sampled: 300 rows for testing

12 batches of up to 25 headlines each
Estimated time: ~0.3 minutes

  Batch 12/12 — OK (25 results)
DONE
  Total rows   : 300
  Relevant     : 255 (85.0%)
  Irrelevant   : 45 (15.0%)

  Category breakdown (relevant only):
    Trade          54
    Policy         40
    Banking        37
    Agriculture    34
    Investment     24
    Energy         15
    Markets        14
    Transport      12
    Forex          9
    Tourism        6
    Inflation      4
    Education      2
    Infrastructure 2
    Taxation       1
    Technology     1

  Sentiment breakdown (relevant only):
    Neutral        114
    Positive       87
    Negative       19

Saved to ../data/processed/tz_headlines_test.csv


In [30]:
# Inspect test results
df_test[df_test["relevant"]==False][["headline", "reason"]]

,headline,reason
33,Tanzania's Taifa Gas licensed to set up plant ...,no clear Tanzanian financial content
38,"EAC plans to host Africa, China – US business ...",no clear Tanzanian financial content
49,Fraud claims wipe $45 billion off India's Adan...,no clear Tanzanian financial content
63,Saudia celebrates Arabia’s 2034 FIFA WC Win wi...,no clear economic theme
135,Don’t be afraid of not taking a well-trodden p...,Don’t be afraid of not taking a well-trodden p...
140,China locks out Kenya from new debt relief deal,China locks out Kenya from new debt relief deal
141,State restore normalcy,State restore normalcy
142,The untold story of the making of Tanganyika -...,The untold story of the making of Tanganyika -...
147,Tasks ahead for new Vodacom Tanzania boss,Tasks ahead for new Vodacom Tanzania boss
149,What future has in store for local startups,What future has in store for local startups


## Section 4: Production Run
Full dataset, `reason` field OFF. Run only when happy with test results.

In [4]:
# Production run — full dataset, no reason field
df_labelled = run_classifier(
    input_csv      = "../../EF-02/data/processed/tz_headlines_clean.csv",
    output_csv     = "../data/processed/tz_headlines_labelled.csv",
    include_reason = False
)

Loaded: 6235 rows from ../../EF-02/data/processed/tz_headlines_clean.csv
Running on full dataset: 6235 rows

250 batches of up to 25 headlines each
Estimated time: ~6.2 minutes

  Batch 34 attempt 1: Validation error — Got 26 results for 25 headlines
  Batch 34 attempt 2: Validation error — Got 26 results for 25 headlines
  Batch 34 attempt 3: Validation error — Got 26 results for 25 headlines
  Batch 34: FAILED after 3 attempts. Filling with None.
  Batch 51 attempt 1: API error — Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kr3gec7df7fry1jtsm9rp8gk` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4596, Requested 1468. Please try again in 640ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  Batch 188 attempt 1: API error — Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama

## Section 5: Retry
Targets only rows where classification failed (relevant is NaN). Uses smaller batches to avoid token overflow.

In [15]:
# Section 5: Retry — targets NaN rows AND hallucinated categories

RETRY_BATCH_SIZE = 10

RETRY_SYSTEM_PROMPT = """You are a financial news classifier for Tanzania.
You return ONLY a valid JSON array. No preamble, no explanation, no markdown fences.
Every object in the array must be complete and the array must be properly closed with ]."""

VALID_CATEGORY_LIST = "Forex | Policy | Banking | Trade | Agriculture | Energy | Transport | Investment | Markets | Tourism | Inflation"

def build_retry_prompt(batch):
    lines = "\n".join(
        f"{i+1}. {item['headline']}"
        for i, item in enumerate(batch)
    )
    return f"""Classify these Tanzanian financial headlines.
Output ONLY a valid JSON array.

Schema: [{{"pos":1,"relevant":true,"category":"Forex","sentiment":"Positive"}}, ...]

RELEVANCE:
relevant true  → Tanzania economy, finance, business, trade, banking, currency, energy, agriculture
relevant false → sports, crime, entertainment, foreign-only news, opinion/lifestyle; if false set category and sentiment to null

CATEGORY — you MUST pick exactly one of these 11 values, no other value is acceptable:
{VALID_CATEGORY_LIST}

Forex(shilling,dollar,reserves) | Policy(BOT,IMF,budget,tax,debt,GDP,NBS,rates) | Banking(banks,loans,fintech,insurance) | Trade(imports,exports,tariffs,ports,AfCFTA) | Agriculture(crops,farming,food) | Energy(TANESCO,fuel,power) | Transport(SGR,rail,roads,logistics) | Investment(FDI,factories,PPP,crowdfunding) | Markets(DSE,CMSA,equity,bonds,turnover) | Tourism(hotels,arrivals) | Inflation(CPI,food/cement price spikes)

SENTIMENT (economic outcome, not tone):
Positive=growth,profit up,shilling firms,easing inflation,launches
Negative=decline,loss,miss target,weakening,rising inflation
Neutral=steady,guidelines,reviews,no clear direction

Headlines:
{lines}"""


def retry_classify_batch(batch, batch_num, total_batches, max_retries=3):
    prompt = build_retry_prompt(batch)
    raw = ""

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": RETRY_SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt}
                ],
                temperature=0,
                max_tokens=1000
            )

            raw    = response.choices[0].message.content.strip()
            clean  = re.sub(r'```(?:json)?|```', '', raw).strip()
            parsed = json.loads(clean)

            if not isinstance(parsed, list):
                raise ValueError(f"Expected list, got {type(parsed)}")
            if len(parsed) != len(batch):
                raise ValueError(f"Got {len(parsed)} results for {len(batch)} headlines")

            for i, result in enumerate(parsed):
                result['id'] = batch[i]['id']
                result.pop('pos', None)

                # If relevant but category is invalid, reject this attempt
                if result.get('relevant') is True:
                    if result.get('category') not in VALID_CATEGORIES:
                        raise ValueError(f"Invalid category on attempt {attempt}: {result.get('category')!r}")
                    if result.get('sentiment') not in VALID_SENTIMENTS:
                        raise ValueError(f"Invalid sentiment on attempt {attempt}: {result.get('sentiment')!r}")

            print(f"  Retry batch {batch_num}/{total_batches} — OK", end="\r", flush=True)
            return parsed

        except json.JSONDecodeError:
            tail = raw[-80:] if len(raw) > 80 else raw
            print(f"  Retry batch {batch_num} attempt {attempt}: JSON cut off. Ends: ...{repr(tail)}")
            time.sleep(2)

        except ValueError as e:
            print(f"  Retry batch {batch_num} attempt {attempt}: {e}")
            time.sleep(2)

        except Exception as e:
            print(f"  Retry batch {batch_num} attempt {attempt}: API error — {e}")
            time.sleep(5)

    # All attempts exhausted — mark as UNCLASSIFIED
    print(f"  Retry batch {batch_num}: FAILED after {max_retries} attempts. Marking UNCLASSIFIED.")
    return [
        {"id": item['id'], "relevant": None, "category": "UNCLASSIFIED", "sentiment": "UNCLASSIFIED"}
        for item in batch
    ]

In [16]:
LABELLED_CSV = "../data/processed/tz_headlines_labelled.csv"

df_full = pd.read_csv(LABELLED_CSV)

# Target both failure types in one mask
mask_nan     = df_full['relevant'].isna()
mask_bad_cat = (
    df_full['relevant'] == True
) & (
    ~df_full['category'].isin(VALID_CATEGORIES)
)
retry_mask = mask_nan | mask_bad_cat

df_retry_input = df_full[retry_mask].copy().reset_index(drop=True)
print(f"NaN rows        : {mask_nan.sum()}")
print(f"Bad category    : {mask_bad_cat.sum()}")
print(f"Total to retry  : {len(df_retry_input)}")

if len(df_retry_input) == 0:
    print("Nothing to retry.")
else:
    df_retry_input['id'] = df_retry_input.index
    rows    = df_retry_input[['id', 'headline']].to_dict('records')
    batches = [rows[i:i+RETRY_BATCH_SIZE] for i in range(0, len(rows), RETRY_BATCH_SIZE)]
    total   = len(batches)
    print(f"{total} retry batches of up to {RETRY_BATCH_SIZE} headlines each\n")

    retry_results = []
    for batch_num, batch in enumerate(batches, start=1):
        results = retry_classify_batch(batch, batch_num, total)
        retry_results.extend(results)
        if batch_num < total:
            time.sleep(SLEEP_SEC)

    df_retry_out = pd.DataFrame(retry_results).drop(columns=['id'])
    original_idx = df_full[retry_mask].index

    df_full.loc[original_idx, ['relevant', 'category', 'sentiment']] = \
        df_retry_out[['relevant', 'category', 'sentiment']].values

    still_nan     = df_full['relevant'].isna().sum()
    still_bad_cat = (
        (df_full['relevant'] == True) &
        (~df_full['category'].isin(VALID_CATEGORIES)) &
        (df_full['category'] != "UNCLASSIFIED")
    ).sum()

    print(f"\nRetry complete.")
    print(f"  Still NaN        : {int(still_nan)}")
    print(f"  Still bad cat    : {int(still_bad_cat)}")
    print(f"  UNCLASSIFIED     : {int((df_full['category'] == 'UNCLASSIFIED').sum())}")

    df_full.to_csv(LABELLED_CSV, index=False)
    print(f"\nSaved to {LABELLED_CSV}")

NaN rows        : 0
Bad category    : 178
Total to retry  : 178
18 retry batches of up to 10 headlines each

  Retry batch 18/18 — OK
Retry complete.
  Still NaN        : 0
  Still bad cat    : 0
  UNCLASSIFIED     : 0

Saved to ../data/processed/tz_headlines_labelled.csv


In [19]:
summary(df_full)


DONE
  Total rows   : 6235
  Relevant     : 5163 (82.8%)
  Irrelevant   : 1072 (17.2%)

  Category breakdown (relevant only):
    Trade          1048
    Policy         939
    Banking        828
    Agriculture    691
    Investment     451
    Markets        359
    Energy         323
    Transport      173
    Forex          163
    Tourism        129
    Inflation      59

  Sentiment breakdown (relevant only):
    Positive       2168
    Neutral        2162
    Negative       405
